In [1]:
import os
import json

from google.generativeai.generative_models import GenerativeModel
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
print("set") if GOOGLE_API_KEY else print("unset")

set


/home/lukas/Programming/uni/threatintel/gemini/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import google.generativeai as genai
genai.configure(api_key=GOOGLE_API_KEY)


for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-1.0-pro-latest
models/gemini-1.0-pro
models/gemini-pro
models/gemini-1.0-pro-001
models/gemini-1.0-pro-vision-latest
models/gemini-pro-vision
models/gemini-1.5-pro-latest
models/gemini-1.5-pro-001
models/gemini-1.5-pro
models/gemini-1.5-pro-exp-0801
models/gemini-1.5-pro-exp-0827
models/gemini-1.5-flash-latest
models/gemini-1.5-flash-001
models/gemini-1.5-flash-001-tuning
models/gemini-1.5-flash
models/gemini-1.5-flash-exp-0827
models/gemini-1.5-flash-8b-exp-0827


In [3]:
from IPython.display import Markdown, display
import textwrap

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [4]:
safety_settings = [
    {
        "category": "HARM_CATEGORY_HARASSMENT",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_HATE_SPEECH",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_SEXUALLY_EXPLICIT",
        "threshold": "BLOCK_NONE",
    },
    {
        "category": "HARM_CATEGORY_DANGEROUS_CONTENT",
        "threshold": "BLOCK_NONE",
    },
]

with open("../resource/stix_specification_plain.txt") as file:
    lines = file.readlines()
system_prompt = "\n".join(lines)
model = genai.GenerativeModel(
    'gemini-1.5-pro', 
    safety_settings=safety_settings, 
    generation_config=genai.GenerationConfig(
        temperature=0.9
    ),
    system_instruction=f"""You are a cyber threat intelligence expert specializing in STIX. You will be provided with STIX 2.1 observables describing potential cyberattacks. Your task is to create corresponding **valid and accurate STIX 2.1 indicators** that effectively detect and characterize these attacks. 

Focus on crafting indicators that are:

* **Specific:** Precisely matching the provided observables.
* **Actionable:** Usable by security tools for detection and response.
* **Context-rich:** Including relevant pattern details, attack motivations, and potential impact.

{system_prompt}
""",
)

In [6]:
with open("../../pattern_matcher/resource/stix/observed/oss/positives_token_theft.json", "r") as file:
    observables_bundle = json.load(file)

prompt = f"""Generate detailed Instructions to create a STIX Pattern which matches the following SDOs as generically as possible:

{str(observables_bundle)}"""


response_indicator = model.generate_content(prompt)
to_markdown(response_indicator.text)
print(response_indicator.text)

> ## STIX 2.1 Indicators from Provided Observables
> 
> These indicators are based on the provided STIX 2.1 bundles and aim to detect similar malicious activity:
> 
> **Indicator 1: Suspicious Service Account Token Fetching**
> 
> ```json
> {
>   "type": "indicator",
>   "id": "indicator--9b0a7471-886b-4a9e-916c-8302f886582d",
>   "spec_version": "2.1",
>   "created": "2024-09-09T19:21:04.000Z",
>   "modified": "2024-09-09T19:21:04.000Z",
>   "name": "Suspicious Service Account Token Fetching from Pod",
>   "description": "Detects attempts to fetch access tokens for service accounts from within a pod, potentially indicating unauthorized access attempts or privilege escalation.",
>   "pattern": "[kubernetes:pod:name = 'pacman/minimi-*' AND kubernetes:container_name = 'gke-metadata-server' AND event:log[string:message LIKE '%Fetching access token for service account%']]",
>   "pattern_type": "stix",
>   "pattern_version": "2.1",
>   "valid_from": "2024-09-09T19:21:04.000Z",
>   "kill_chain_phases": [
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "credential-access"
>     },
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "privilege-escalation"
>     }
>   ],
>   "indicator_types": [
>     "malicious-activity"
>   ],
>   "confidence": "80",
>   "severity": "medium"
> }
> ```
> 
> **Indicator 2: Successful New Entry Loading in Suspicious Pod**
> 
> ```json
> {
>   "type": "indicator",
>   "id": "indicator--4a148181-175a-4d3b-883d-734c8c73487e",
>   "spec_version": "2.1",
>   "created": "2024-09-09T19:21:04.000Z",
>   "modified": "2024-09-09T19:21:04.000Z",
>   "name": "Successful New Entry Loading in Suspicious Pod",
>   "description": "Detects successful loading of new entries in a pod potentially associated with malicious activity, indicating possible data exfiltration or command and control communication.",
>   "pattern": "[kubernetes:pod:name = 'pacman/minimi-*' AND kubernetes:container_name = 'gke-metadata-server' AND event:log[string:message LIKE '%Loading new entry succeeded%']]",
>   "pattern_type": "stix",
>   "pattern_version": "2.1",
>   "valid_from": "2024-09-09T19:21:04.000Z",
>   "kill_chain_phases": [
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "command-and-control"
>     },
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "exfiltration"
>     }
>   ],
>   "indicator_types": [
>     "malicious-activity"
>   ],
>   "confidence": "70",
>   "severity": "low"
> }
> ```
> 
> **Indicator 3: Suspicious HTTP Request to Metadata Server**
> 
> ```json
> {
>   "type": "indicator",
>   "id": "indicator--a73a0d56-6e4f-4a43-b99b-002d5c5a208a",
>   "spec_version": "2.1",
>   "created": "2024-09-09T19:21:04.000Z",
>   "modified": "2024-09-09T19:21:04.000Z",
>   "name": "Suspicious HTTP Request to Metadata Server for Service Account Token",
>   "description": "Detects suspicious HTTP requests targeting the Google metadata server aiming to retrieve service account tokens, often associated with SSRF attacks or attempts to gain unauthorized cloud resource access.",
>   "pattern": "[network-traffic:http_request:method = 'GET' AND network-traffic:http_request:uri LIKE '%/computeMetadata/v1/instance/service-accounts/default/token%' AND network-traffic:src_ref.value = '10.1.2.8']",
>   "pattern_type": "stix",
>   "pattern_version": "2.1",
>   "valid_from": "2024-09-09T19:21:04.000Z",
>   "kill_chain_phases": [
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "reconnaissance"
>     },
>     {
>       "kill_chain_name": "mitre-attack",
>       "phase_name": "credential-access"
>     }
>   ],
>   "indicator_types": [
>     "malicious-activity"
>   ],
>   "confidence": "90",
>   "severity": "high"
> }
> ```
> 
> **Notes:**
> 
> * The `confidence` and `severity` levels are subjective and should be adjusted based on your specific environment and threat model.
> * The provided indicators are examples and may need to be adapted to your specific needs and detection capabilities.
> * Consider enriching these indicators with additional context, such as TTPs, known threat actors, and impacted platforms.
> * It's crucial to continuously monitor and update your indicators to stay ahead of evolving threats. 
